In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week5-assignment-2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
## path : /public/trendytech/retail_db/products

In [3]:
! hadoop fs -ls /public/trendytech/retail_db/products

Found 1 items
-rw-r--r--   3 itv005857 supergroup     174155 2023-04-26 16:47 /public/trendytech/retail_db/products/part-00000


In [4]:
## Cols ProductID, Category, ProductName, Description, Price,ImageURL

In [5]:
!hadoop fs -head /public/trendytech/retail_db/products/part-00000

1,2,Quest Q64 10 FT. x 10 FT. Slant Leg Instant U,,59.98,http://images.acmesports.sports/Quest+Q64+10+FT.+x+10+FT.+Slant+Leg+Instant+Up+Canopy
2,2,Under Armour Men's Highlight MC Football Clea,,129.99,http://images.acmesports.sports/Under+Armour+Men%27s+Highlight+MC+Football+Cleat
3,2,Under Armour Men's Renegade D Mid Football Cl,,89.99,http://images.acmesports.sports/Under+Armour+Men%27s+Renegade+D+Mid+Football+Cleat
4,2,Under Armour Men's Renegade D Mid Football Cl,,89.99,http://images.acmesports.sports/Under+Armour+Men%27s+Renegade+D+Mid+Football+Cleat
5,2,Riddell Youth Revolution Speed Custom Footbal,,199.99,http://images.acmesports.sports/Riddell+Youth+Revolution+Speed+Custom+Football+Helmet
6,2,Jordan Men's VI Retro TD Football Cleat,,134.99,http://images.acmesports.sports/Jordan+Men%27s+VI+Retro+TD+Football+Cleat
7,2,Schutt Youth Recruit Hybrid Custom Football H,,99.99,http://images.acmesports.sports/Schutt+Youth+Recruit+Hybrid+Custom+Football+Helmet+2014
8,2,Nike Men's Vapor Ca

In [6]:
products_df = spark.read.csv('/public/trendytech/retail_db/products/part-00000',header = "false",inferSchema = "true")

In [7]:
spark.sql("use itv024128")

""


In [8]:
products_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: string (nullable = true)



In [9]:
products_df.show(4)

+---+---+--------------------+----+------+--------------------+
|_c0|_c1|                 _c2| _c3|   _c4|                 _c5|
+---+---+--------------------+----+------+--------------------+
|  1|  2|Quest Q64 10 FT. ...|null| 59.98|http://images.acm...|
|  2|  2|Under Armour Men'...|null|129.99|http://images.acm...|
|  3|  2|Under Armour Men'...|null| 89.99|http://images.acm...|
|  4|  2|Under Armour Men'...|null| 89.99|http://images.acm...|
+---+---+--------------------+----+------+--------------------+
only showing top 4 rows



In [10]:
products_df_1 = products_df.withColumnRenamed("_c0","ProductID") \
.withColumnRenamed("_c1","Category") \
.withColumnRenamed("_c2","ProductName") \
.withColumnRenamed("_c3","Description") \
.withColumnRenamed("_c4","Price") \
.withColumnRenamed("_c5","ImageURL")

In [11]:
products_df_1.printSchema()

root
 |-- ProductID: integer (nullable = true)
 |-- Category: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- ImageURL: string (nullable = true)



In [12]:
products_df_1.show(4)

+---------+--------+--------------------+-----------+------+--------------------+
|ProductID|Category|         ProductName|Description| Price|            ImageURL|
+---------+--------+--------------------+-----------+------+--------------------+
|        1|       2|Quest Q64 10 FT. ...|       null| 59.98|http://images.acm...|
|        2|       2|Under Armour Men'...|       null|129.99|http://images.acm...|
|        3|       2|Under Armour Men'...|       null| 89.99|http://images.acm...|
|        4|       2|Under Armour Men'...|       null| 89.99|http://images.acm...|
+---------+--------+--------------------+-----------+------+--------------------+
only showing top 4 rows



In [13]:
products_df_1.createOrReplaceTempView("products")

In [14]:
# 2.1. Find the total number of products in the given dataset.

In [15]:
products_df_1.select("ProductID").distinct().count()

1345

In [16]:
##2.2. Find the number of unique categories of products in the given dataset.

In [17]:
products_df_1.select("Category").distinct().count()

55

In [18]:
## 2.3. Find the top 5 most expensive products based on their price, along with their product name, category, and image URL.

In [19]:
products_df_2 = products_df_1.select("ProductName","Category","ImageURL","Price").orderBy("Price",ascending=False)

In [20]:
products_df_2.show(5)

+--------------------+--------+--------------------+-------+
|         ProductName|Category|            ImageURL|  Price|
+--------------------+--------+--------------------+-------+
| SOLE E35 Elliptical|      10|http://images.acm...|1999.99|
|  SOLE F85 Treadmill|      10|http://images.acm...|1799.99|
|  SOLE F85 Treadmill|      22|http://images.acm...|1799.99|
|  SOLE F85 Treadmill|       4|http://images.acm...|1799.99|
|"Spalding Beast 6...|      47|http://images.acm...|1099.99|
+--------------------+--------+--------------------+-------+
only showing top 5 rows



In [21]:
## 2.4. Find the number of products in each category that have a price greater than $100. Display the results in a tabular format that shows the category 
## name and the number of products that satisfy the condition.

In [22]:
cat_count = products_df_1.filter("Price > 100").select("Category","ProductID").groupBy("Category").count()

In [23]:
cat_count.show()

+--------+-----+
|Category|count|
+--------+-----+
|      31|   17|
|      53|   16|
|      34|   15|
|      44|    9|
|      12|    3|
|      22|    4|
|      47|   10|
|      52|    5|
|      13|    1|
|       6|    5|
|      16|   11|
|       3|    5|
|      20|    7|
|      57|    6|
|      54|    6|
|      48|   17|
|       5|   11|
|      19|   13|
|      41|   11|
|      43|   23|
+--------+-----+
only showing top 20 rows



In [24]:
## 2.5. What are the product names and prices of products that have a price greater than $200 and belong to category 5?

In [25]:
cat5_count = products_df_1.filter("Price > 200 and Category = 5").select("ProductName","Price")

In [26]:
cat5_count.show()

+--------------------+------+
|         ProductName| Price|
+--------------------+------+
|"Goaliath 54"" In...|499.99|
|Fitness Gear 300 ...|209.99|
|Teeter Hang Ups N...|299.99|
+--------------------+------+



In [27]:
## Find the total number of products in the given dataset.

In [28]:
spark.sql("select *  from products limit 5").show()

+---------+--------+--------------------+-----------+------+--------------------+
|ProductID|Category|         ProductName|Description| Price|            ImageURL|
+---------+--------+--------------------+-----------+------+--------------------+
|        1|       2|Quest Q64 10 FT. ...|       null| 59.98|http://images.acm...|
|        2|       2|Under Armour Men'...|       null|129.99|http://images.acm...|
|        3|       2|Under Armour Men'...|       null| 89.99|http://images.acm...|
|        4|       2|Under Armour Men'...|       null| 89.99|http://images.acm...|
|        5|       2|Riddell Youth Rev...|       null|199.99|http://images.acm...|
+---------+--------+--------------------+-----------+------+--------------------+



In [29]:
spark.sql("select count(*) from products")

count(1)
1345


In [30]:
## Find the number of unique categories of products in the given dataset.

In [31]:
spark.sql("select count(distinct category ) from products ")

count(DISTINCT category)
55


In [32]:
## 2.3. Find the top 5 most expensive products based on their price, along with their product name, category, and image URL.

In [33]:
spark.sql("select ProductName,Category,ImageURL,Price from products order by Price desc limit 5")

ProductName,Category,ImageURL,Price
SOLE E35 Elliptical,10,http://images.acm...,1999.99
SOLE F85 Treadmill,4,http://images.acm...,1799.99
SOLE F85 Treadmill,10,http://images.acm...,1799.99
SOLE F85 Treadmill,22,http://images.acm...,1799.99
"""Spalding Beast 6...",47,http://images.acm...,1099.99


In [34]:
## 2.4. Find the number of products in each category that have a price greater than $100. Display the results in a tabular format that shows the category 
## name and the number of products that satisfy the condition.

In [35]:
spark.sql("select category, count(*) as number_of_prod from products where price > 100 group by category")

category,number_of_prod
31,17
53,16
34,15
44,9
12,3
22,4
47,10
52,5
13,1
6,5


In [36]:
## 2.5. What are the product names and prices of products that have a price greater than $200 and belong to category 5?

In [37]:
spark.sql("select ProductName,Price from products where price > 200 and category = 5")

ProductName,Price
"""Goaliath 54"""" In...",499.99
Fitness Gear 300 ...,209.99
Teeter Hang Ups N...,299.99
